# 2장 3강: MSE와 Cross Entropy의 수학적 원리 — 실습문제

## 실습 목표

- 실제값에서 예측값을 빼고, 제곱한 뒤 평균을 내어 MSE를 구합니다.
- 실제 구매 여부에 맞는 확률을 사용해 BCE를 계산합니다.
- 실제 상품 종류에 해당하는 확률을 골라 Categorical CE를 계산합니다.
- 주어진 Softmax 함수로 점수를 확률로 바꾸고 CE를 구합니다.
- 교안에서 MSE·BCE·CE가 NLL과 연결되는 뜻을 쉬운 설명으로 확인합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 별도의 데이터 파일은 필요하지 않습니다. 데이터와 계산 방법은 제공된 「2장 3강: MSE와 Cross Entropy의 수학적 원리」의 본문 예제와 연습문제를 사용합니다.

회귀의 오차는 교안에 맞추어 **실제값 − 예측값**으로 계산합니다. 이진 분류는 **0=미구매, 1=구매**, 상품 분류는 **0=의류, 1=식품, 2=가전**입니다. 같은 숫자라도 문제에 따라 뜻이 다르므로 먼저 표의 제목을 확인하세요.

## 실습 구성과 교안 대응

| 구분 | 계산할 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | 두 금액 예측 모델의 오차·제곱오차·SSE·MSE | 1절 본문: 실제값 [10,20,30] 예제 |
| 필수 1 / 문제 1-2 | 고객별 BCE와 평균 BCE 비교 | 2절 본문: 고객 4명의 두 모델 비교 |
| 필수 2 / 문제 2-1 | 실제 정답 열 선택, 상품별 CE와 평균 CE | 3절 본문: 의류·식품·가전 상품 3개 예제 |
| 필수 2 / 문제 2-2 | 제공된 Softmax 함수로 확률과 CE 계산 | 3절 본문: Logit [2,1,0], [0,2,1], [1,0,3] |
| 과제 / 문제 3-1 | 새 금액 예측의 MSE 복습 | 연습문제 1의 MSE 계산 부분 |
| 과제 / 문제 3-2 | 새 고객 데이터의 BCE 복습 | 연습문제 2의 고객별·평균 BCE 계산 부분 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 본문 예제와 연습문제의 일부를 기초 실습 4문제와 복습 과제 2문제로 재구성했습니다. 과제에서도 앞에서 익힌 계산식과 교안에 있는 숫자를 사용합니다.

## 진행 안내

1. NumPy를 불러오는 셀부터 실행합니다.
2. 표를 읽고 실제값·예측값 또는 실제 정답·예측 확률을 구분합니다.
3. 제곱은 각 오차에 먼저 적용하고, 그다음 평균을 구합니다.
4. 분류에서는 **실제 정답에 부여한 확률**을 찾은 다음 음의 로그를 적용합니다.
5. 제공된 함수와 확률 선택 코드는 그대로 사용해도 됩니다. 공식을 유도하거나 함수를 외워서 작성할 필요는 없습니다.
6. 계산 중간값은 반올림하지 않고 출력에만 `round()`·`np.round()`를 사용합니다.
7. 각 문제의 Q1~Q4에 계산값과 쉬운 문장으로 답하세요.

## 먼저 읽는 용어와 수식 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 손실 함수 / Loss Function | 모델의 예측을 정답과 비교하여 손실값으로 계산하는 규칙 |
| 회귀 / 분류 | 금액처럼 수치를 예측 / 정해진 종류를 예측 |
| y, y_true | 실제 정답. 금액인지 구매 여부인지 상품 종류인지 문제에 따라 확인 |
| ŷ, y_pred | 모델이 예측한 수치. ŷ는 ‘와이 햇’으로 읽음 |
| 오차 / errors | 실제값에서 예측값을 뺀 수 |
| 제곱 / ** 2 | 같은 수를 두 번 곱함. (−2)²=(−2)×(−2)=4 |
| SSE | 제곱오차를 모두 더한 값 |
| MSE | 제곱오차의 평균. SSE를 데이터 개수로 나눈 값 |
| p / 1−p | 이진 분류의 구매 확률 / 미구매 확률 |
| 클래스 / 인덱스 | 종류의 이름 / 배열에서의 위치 번호. 첫 위치는 0 |
| BCE | Binary Cross Entropy. 이 자료의 구매·미구매 분류에서 쓰는 손실 |
| CE | 이 자료에서는 Categorical Cross Entropy. 상품이 한 종류에만 속하는 분류의 손실 |
| One-Hot | 정답 클래스의 위치만 1, 나머지 위치는 0으로 표시한 배열 |
| Logit / Softmax | 확률로 바꾸기 전 점수 / 여러 점수를 합이 1인 확률로 바꾸는 함수 |
| n / Σ / ≈ | 데이터 개수 / 여러 값을 더하라는 기호 / 대략 같음 |
| ln / np.log | 자연로그 / 자연로그를 계산하는 NumPy 코드 |
| −ln(p) | 먼저 p의 자연로그를 구한 뒤 부호를 바꿈 |
| NLL | Negative Log-Likelihood. 우도의 로그에 마이너스를 붙인 값 |
| Mean / Total | 평균 / 합계 |

**로그를 처음 보는 경우:** 로그는 ‘어떤 수를 몇 제곱하면 주어진 값이 되는가’를 묻는 계산입니다. 자연로그는 약 2.718인 수 e를 기준으로 합니다. 이번 실습에서는 로그값을 손으로 구하지 않고 `np.log()`를 사용합니다. 예를 들어 `np.log(0.8)`은 약 −0.2231이고, `-np.log(0.8)`은 부호를 바꾼 약 0.2231입니다. 0과 1 사이 확률에 로그를 적용하면 음수이므로 앞의 마이너스를 빠뜨리지 마세요.

**손실 비교의 기준:** 같은 데이터·같은 손실 함수에서 값이 작은 쪽을 선택합니다. MSE와 BCE처럼 서로 다른 손실값을 숫자 크기만으로 비교하지 않습니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 배열을 계산하는 도구이고, `np`는 NumPy를 짧게 부르는 이름입니다.

In [16]:
import numpy as np

---

## 필수 1. 회귀와 이진 분류의 손실 계산하기

### 문제 1-1. 오차를 제곱하고 평균 내어 MSE 비교하기

#### 문제 설명

교안 1절 본문의 구매 금액 예측입니다. 단위는 **만 원**입니다.

| 고객 | 실제값 | 모델 A 예측 | 모델 B 예측 |
|---|---:|---:|---:|
| 첫째 | 10 | 12 | 9 |
| 둘째 | 20 | 18 | 24 |
| 셋째 | 30 | 27 | 33 |

오차는 실제값−예측값, 제곱오차는 오차×오차입니다. SSE는 제곱오차의 합, MSE는 제곱오차의 평균입니다.

#### 요구사항

1. 실제값과 두 모델의 예측값을 각각 배열로 만드세요.
2. 두 모델의 오차와 제곱오차를 계산해 출력하세요.
3. `np.sum()`으로 SSE, `np.mean()`으로 MSE를 구하세요. MSE는 소수점 아래 세 자리로 표시합니다.
4. Q1~Q4에 답하세요.

#### 작성 도움말

```python
errors = y_true - y_pred
squared_errors = errors ** 2
sse = np.sum(squared_errors)
mse = np.mean(squared_errors)
```

먼저 각 오차를 제곱해야 합니다. 오차를 먼저 평균한 뒤 제곱하는 계산과 다릅니다. 위의 변수 이름을 두 모델에 맞게 구분해서 사용하세요.

#### 해석 질문

**Q1.** 두 모델의 오차와 제곱오차는 각각 무엇인가요?

**Q2.** SSE와 MSE는 각각 얼마이며, 어느 모델을 선택하나요?

**Q3.** 모델 A의 첫 오차 −2는 실제보다 높게 예측했다는 뜻인가요, 낮게 예측했다는 뜻인가요?

**Q4.** MSE 5.667을 평균 5.667만 원 틀렸다고 해석해도 되나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [30]:
# 문제 1-1. 오차를 제곱하고 평균 내어 MSE 비교하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1. 실제값과 두 모델의 예측값을 배열로 만드세요. (단위: 만 원)
y_true = np.array([10, 20, 30])
pred_a = np.array([12, 18, 27])
pred_b = np.array([9, 24, 33])


# 2. 두 모델의 오차와 제곱오차를 출력하세요.
errors_a = y_true - pred_a
squared_errors_a = errors_a ** 2
errors_b = y_true - pred_b
squared_errors_b = errors_b ** 2

print("모델 A 오차 :", errors_a)
print("모델 A 제곱오차 :", squared_errors_a)
print("모델 B 오차 :", errors_b)
print("모델 B 제곱오차 :", squared_errors_b)


# 3. SSE와 MSE를 구하세요.
sse_a = np.sum(squared_errors_a)
mse_a = np.mean(squared_errors_a)
sse_b = np.sum(squared_errors_b)
mse_b = np.mean(squared_errors_b)

print("모델 A SSE :", sse_a, "/ MSE :", round(mse_a, 3))
print("모델 B SSE :", sse_b, "/ MSE :", round(mse_b, 3))

모델 A 오차 : [-2  2  3]
모델 A 제곱오차 : [4 4 9]
모델 B 오차 : [ 1 -4 -3]
모델 B 제곱오차 : [ 1 16  9]
모델 A SSE : 17 / MSE : 5.667
모델 B SSE : 26 / MSE : 8.667


#### 답변 작성란

**Q1. 두 모델의 오차와 제곱오차는 각각 무엇인가요?**

- 모델 A  
오차 : [10 − 12, 20 − 18, 30 − 27]  
-> [−2, 2, 3]  
제곱오차 : [(−2)², 2², 3²]  
-> [4, 4, 9]  
- 모델 B  
오차 : [10 − 9, 20 − 24, 30 − 33]  
-> [1, −4, −3]  
제곱오차 : [1², (−4)², (−3)²]  
-> [1, 16, 9]  

**Q2. SSE와 MSE는 각각 얼마이며, 어느 모델을 선택하나요?**

- **모델 A**  
모델 A  
SSE : 4 + 4 + 9 = 17  
MSE : 17 / 3 ≈ 5.667  
모델 B  
SSE : 1 + 16 + 9 = 26  
MSE : 26 / 3 ≈ 8.667  
-> 동일한 데이터의 MSE가 더 작은 모델 A를 선택하는 것이 합리적  

**Q3. 모델 A의 첫 오차 −2는 실제보다 높게 예측했다는 뜻인가요, 낮게 예측했다는 뜻인가요?**

- 실제보다 **높게** 예측했다는 뜻  
-> 실제는 10만원인데 예측은 12만원이니까 실제보다 2만원 높음  
오차 = 실제값 − 예측값 = 10 − 12 = −2  
-> 예측값(12)이 실제값(10)보다 커서 오차가 음수가 됨  
-> 실제보다 2만 원 많게 예측  

**Q4. MSE 5.667을 평균 5.667만 원 틀렸다고 해석해도 되나요?**

- 아니오  
MSE는 오차를 제곱한 뒤 평균한 값이므로 단위가 만 원이 아니라 (만 원)²  
-> 실제 금액 단위로 해석하려면 루트를 씌운 RMSE를 사용  
-> RMSE = √5.667 ≈ 2.380 -> 평균적으로 약 2.38만 원 정도 틀렸다고 해석

### 문제 1-2. 실제 구매 여부에 맞추어 BCE 계산하기

#### 문제 설명

교안 2절 본문의 두 모델입니다. 실제 정답은 1=구매, 0=미구매이며 p는 항상 구매 확률입니다.

| 고객 | 실제 정답 y | 모델 A의 p | 모델 B의 p |
|---|---:|---:|---:|
| 첫째 | 1 | 0.9 | 0.9 |
| 둘째 | 0 | 0.2 | 0.2 |
| 셋째 | 1 | 0.6 | 0.8 |
| 넷째 | 0 | 0.7 | 0.1 |

고객 한 명의 BCE는 `−[y×ln(p)+(1−y)×ln(1−p)]`입니다. y=1이면 −ln(p), y=0이면 −ln(1−p)로 계산합니다.

#### 요구사항

1. 실제 정답과 모델별 구매 확률을 배열로 만드세요.
2. 아래 함수를 그대로 사용하여 두 모델의 고객별 BCE를 구하세요.
3. 각 손실 배열에 `np.mean()`을 적용해 평균 BCE를 구하세요. 고객별·평균 손실을 소수점 아래 세 자리로 출력합니다.
4. Q1~Q4에 답하세요.

#### 제공 함수와 작성 도움말

```python
def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))
```

이 함수는 고객별 손실 배열을 반환합니다. 평균은 함수 밖에서 구합니다. 예를 들어 실제 정답이 0이고 구매 확률이 0.7이면, 실제 정답인 미구매의 확률 1−0.7=0.3을 사용합니다. 제공된 확률은 모두 0보다 크고 1보다 작아 이 식으로 계산할 수 있습니다.

#### 해석 질문

**Q1.** 모델 A가 실제 정답에 부여한 확률은 각각 얼마인가요?

**Q2.** 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?

**Q3.** 미구매한 넷째 고객의 모델 A 손실에 왜 0.7이 아니라 0.3을 사용하나요?

**Q4.** 어느 모델을 선택하며, 평균 손실이 작아진 이유는 무엇인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [ ]:
# 문제 1-2. 실제 구매 여부에 맞추어 BCE 계산하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1. 실제 정답과 모델별 구매 확률을 배열로 만드세요.
y = np.array([1, 0, 1, 0])
p_a = np.array([0.9, 0.2, 0.6, 0.7])
p_b = np.array([0.9, 0.2, 0.8, 0.1])


# 2. 두 모델의 고객별 BCE를 구하세요.
def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))

bce_a = binary_cross_entropy(y, p_a) # 이진 분류
bce_b = binary_cross_entropy(y, p_b)
print("모델 A 고객별 BCE :", np.round(bce_a, 3))
print("모델 B 고객별 BCE :", np.round(bce_b, 3))


# 3. 평균 BCE를 구하세요.
mean_a = np.mean(bce_a)
mean_b = np.mean(bce_b)
print("모델 A 평균 BCE :", round(mean_a, 3))
print("모델 B 평균 BCE :", round(mean_b, 3))

모델 A 고객별 BCE : [0.105 0.223 0.511 1.204]
모델 B 고객별 BCE : [0.105 0.223 0.223 0.105]
모델 A 평균 BCE : 0.511
모델 B 평균 BCE : 0.164


#### 답변 작성란

**Q1. 모델 A가 실제 정답에 부여한 확률은 각각 얼마인가요?**

- 답변: [0.9, 0.8, 0.6, 0.3]  
첫째 : 구매(1) -> p = 0.9  
둘째 : 미구매(0) -> 1 − 0.2 = 0.8  
셋째 : 구매(1) -> p = 0.6  
넷째 : 미구매(0) -> 1 − 0.7 = 0.3  
-> 실제 값이 0인 2,4번째는 구매확률 p대신 1-p를 사용

**Q2. 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?**

- 답변:  
모델 A  
고객별 BCE : [−ln(0.9), −ln(0.8), −ln(0.6), −ln(0.3)] = [0.105, 0.223, 0.511, 1.204]  
평균 BCE : (0.105 + 0.223 + 0.511 + 1.204) / 4 ≈ 0.511  
모델 B  
고객별 BCE : [−ln(0.9), −ln(0.8), −ln(0.8), −ln(0.9)] = [0.105, 0.223, 0.223, 0.105]  
평균 BCE : (0.105 + 0.223 + 0.223 + 0.105) / 4 ≈ 0.164  

**Q3. 미구매한 넷째 고객의 모델 A 손실에 왜 0.7이 아니라 0.3을 사용하나요?**

- 답변: 넷째 고객의 실제 정답이 미구매(0)이기 때문  
0.7은 모델 A가 준 **구매** 확률이고, 실제 정답은 미구매였기 때문. 미구매의 확률이은 1 − 0.7 = 0.3  
-> 0.7을 쓰게되면 손실을 구매가 정답인 경우의 손실을 구하기 때문에 0.3을 쓰는 것이 더 적절하다.  

**Q4. 어느 모델을 선택하며, 평균 손실이 작아진 이유는 무엇인가요?**

- 답변: **모델 B**  
모델 A 평균 BCE 0.511 > 모델 B 평균 BCE 0.164  
-> B는 셋째의 실제 구매 확률을 0.6에서 0.8로, 넷째의 시제 미구매 확률을 0.3에서 0.9로 높임  
-> 실제 정답에 더 높은 확률을 주어서 손실을 줄였음

---

## 필수 2. 다중 클래스 분류의 손실 계산하기

### 문제 2-1. 실제 정답 클래스의 확률로 CE 계산하기

#### 문제 설명

교안 3절의 상품 3개 예제입니다. 클래스 순서는 **의류 0, 식품 1, 가전 2**입니다. 한 상품의 실제 정답은 한 종류입니다.

| 상품 | 실제 정답 | 의류 확률 | 식품 확률 | 가전 확률 |
|---|---|---:|---:|---:|
| A | 의류 | 0.8 | 0.1 | 0.1 |
| B | 식품 | 0.2 | 0.6 | 0.2 |
| C | 가전 | 0.1 | 0.3 | 0.6 |

상품 하나의 `CE = −ln(실제 정답 클래스의 확률)`입니다. 평균 CE는 상품별 CE를 더하고 상품 수로 나눕니다.

#### 요구사항

1. 정답 배열 `y_true = np.array([0,1,2])`와 표의 3행 3열 확률 배열 `y_prob`를 만드세요.
2. 아래 교안의 코드로 상품별 실제 정답 확률을 골라 출력하세요.
3. `-np.log(true_class_prob)`로 상품별 CE를 구하고 `np.mean()`으로 평균을 구하세요. 손실은 소수점 아래 세 자리로 출력합니다.
4. Q1~Q4에 답하세요.

#### 제공 코드와 읽는 방법

```python
rows = np.arange(len(y_true))
true_class_prob = y_prob[rows, y_true]
```

rows는 상품의 행 번호 [0,1,2]입니다. y_true는 그 상품의 정답 열 번호입니다. 두 배열의 같은 위치를 짝지어 (0행,0열), (1행,1열), (2행,2열)을 고릅니다. 인덱스는 0부터 셉니다. 코드는 그대로 사용해도 됩니다.

#### 해석 질문

**Q1.** 각 상품에서 어느 행·열을 선택하며, 선택한 확률은 얼마인가요?

**Q2.** 상품별 CE와 평균 CE는 각각 얼마인가요?

**Q3.** 평균 CE에서 3으로 나누는 이유는 무엇인가요?

**Q4.** CE에 사용할 확률은 가장 큰 예측 확률인가요, 실제 정답 클래스의 확률인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [32]:
# 문제 2-1. 실제 정답 클래스의 확률로 CE 계산하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1. 정답 배열과 확률 배열을 만드세요. (클래스 순서: 의류 0, 식품 1, 가전 2)
y_true = np.array([0, 1, 2])          # 상품 A 의류, B 식품, C 가전
y_prob = np.array([
    [0.8, 0.1, 0.1],  # 상품 A
    [0.2, 0.6, 0.2],  # 상품 B
    [0.1, 0.3, 0.6]   # 상품 C
    ])


# 2. 상품별 실제 정답 확률을 골라 출력하세요.
rows = np.arange(len(y_true))
true_class_prob = y_prob[rows, y_true]
print("실제 정답 클래스 확률 :", true_class_prob)


# 3. 상품별 CE와 평균 CE를 구하세요.
ce_each = -np.log(true_class_prob)
ce_mean = np.mean(ce_each)
print("상품별 CE :", np.round(ce_each, 3))
print("평균 CE :", round(ce_mean, 3))

실제 정답 클래스 확률 : [0.8 0.6 0.6]
상품별 CE : [0.223 0.511 0.511]
평균 CE : 0.415


#### 답변 작성란

**Q1. 각 상품에서 어느 행·열을 선택하며, 선택한 확률은 얼마인가요?**

- 답변: [0.8, 0.6, 0.6]  
상품 A : 0행, 0열 (정답 의류) -> 0.8  
상품 B : 1행, 1열 (정답 식품) -> 0.6  
상품 C : 2행, 2열 (정답 가전) -> 0.6  
-> rows [0, 1, 2]와 y_true [0, 1, 2]를 같은 위치끼리 짝지어 선택  

**Q2. 상품별 CE와 평균 CE는 각각 얼마인가요?**

- 답변:  
상품별 CE : [−ln(0.8), −ln(0.6), −ln(0.6)] = [0.223, 0.511, 0.511]  
평균 CE : (0.223 + 0.511 + 0.511) / 3 ≈ 0.415  

**Q3. 평균 CE에서 3으로 나누는 이유는 무엇인가요?**

- 답변: 손실을 계산하려는 상품이 3개(A, B, C)이기 때문  
-> 상품별 CE 세 개의 평균을 구하는 계산이기 때문  
평균 CE는 상품별 CE를 모두 더한 뒤 상품 수로 나눔  
-> 클래스 수(의류·식품·가전 3개)가 아니라 상품 수로 나누는 것이며, 이 문제는 우연히 둘 다 3  

**Q4. CE에 사용할 확률은 가장 큰 예측 확률인가요, 실제 정답 클래스의 확률인가요?**

- 답변: **실제 정답 클래스의 확률**  
가장 큰 예측 확률은 모델이 어느 클래스를 선택했는지 알려다.  
-> 그 클래스가 실제 정답과 다를 수 있기 때문에 CE에서는 Y_true가 가리키는 정답 열을 사용한다. 

-> 이번 문제는 세 상품 모두 정답 클래스의 확률이 가장 커서 두 값이 같지만, 모델이 틀린 클래스에 가장 큰 확률을 준 경우에도 CE에는 정답 클래스의 확률을 사용해야 함  
-> 예: 상품 C의 정답이 식품이었다면 가장 큰 0.6(가전)이 아니라 0.3(식품)을 사용 -> −ln(0.3) ≈ 1.204

### 문제 2-2. 주어진 Softmax 함수로 확률과 CE 계산하기

#### 문제 설명

교안 3절의 점수(Logit) 예제입니다. 클래스 순서는 의류 0, 식품 1, 가전 2입니다.

| 상품 | 의류 점수 | 식품 점수 | 가전 점수 | 실제 정답 인덱스 |
|---|---:|---:|---:|---:|
| A | 2 | 1 | 0 | 0 |
| B | 0 | 2 | 1 | 1 |
| C | 1 | 0 | 3 | 2 |

점수는 아직 확률이 아닙니다. 아래 **교안의 Softmax 함수를 그대로 사용**하여 확률로 바꾸세요.

```python
def softmax(z):
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)
```

#### 요구사항

1. 점수 행렬 logits와 정답 배열 y_true를 만들고, 제공된 함수를 작성하세요.
2. `probabilities = softmax(logits)`로 확률을 구하고, `np.sum(probabilities, axis=1)`로 상품별 확률 합을 확인하세요.
3. `probabilities[np.arange(len(y_true)), y_true]`로 실제 정답 확률을 선택하세요. 상품별 CE와 평균 CE를 구하세요.
4. 확률과 손실은 소수점 아래 세 자리로 출력하고 Q1~Q4에 답하세요.

#### 작성 도움말

`axis=1`은 각 상품 행 안의 클래스 방향으로 계산하라는 뜻입니다. 정답 확률을 선택한 다음의 CE 계산은 문제 2-1과 같습니다. 손실 계산에는 반올림 전 확률을 사용하세요.

#### 해석 질문

**Q1.** Softmax 결과와 상품별 확률 합은 무엇인가요?

**Q2.** 실제 정답 확률, 상품별 CE, 평균 CE는 각각 얼마인가요?

**Q3.** 세 상품 모두 정답을 가장 높게 예측했는데 C의 손실이 더 작은 이유는 무엇인가요?

**Q4.** 점수 [2,1,0]을 곧바로 CE의 확률 자리에 넣어도 되나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [34]:
# 문제 2-2. 주어진 Softmax 함수로 확률과 CE 계산하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1. 점수 행렬 logits와 정답 배열 y_true를 만들고, 제공된 함수를 작성하세요.
logits = np.array([
    [2, 1, 0],  # 상품 A
    [0, 2, 1],  # 상품 B
    [1, 0, 3]   # 상품 C
    ])
y_true = np.array([0, 1, 2])   # 의류, 식품, 가전

def softmax(z):
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)


# 2. 확률을 구하고 상품별 확률 합을 확인하세요.
probabilities = softmax(logits)
print("== Softmax 확률 ==")
print(np.round(probabilities, 3))
print("상품별 확률 합 :", np.sum(probabilities, axis=1))


# 3. 실제 정답 확률을 선택하고 상품별 CE와 평균 CE를 구하세요.
true_class_prob = probabilities[np.arange(len(y_true)), y_true]
ce_each = -np.log(true_class_prob)    # 반올림 전 확률 사용
ce_mean = np.mean(ce_each)

print("\n실제 정답 확률 :", np.round(true_class_prob, 3))
print("상품별 CE :", np.round(ce_each, 3))
print("평균 CE :", round(ce_mean, 3))

== Softmax 확률 ==
[[0.665 0.245 0.09 ]
 [0.09  0.665 0.245]
 [0.114 0.042 0.844]]
상품별 확률 합 : [1. 1. 1.]

실제 정답 확률 : [0.665 0.665 0.844]
상품별 CE : [0.408 0.408 0.17 ]
평균 CE : 0.328


#### 답변 작성란

**Q1. Softmax 결과와 상품별 확률 합은 무엇인가요?**

- 답변:  
상품 A : [0.665, 0.245, 0.090]  
상품 B : [0.090, 0.665, 0.245]  
상품 C : [0.114, 0.042, 0.844]  
-> 상품별 확률 합은 모두 1  

**Q2. 실제 정답 확률, 상품별 CE, 평균 CE는 각각 얼마인가요?**

- 답변:  
실제 정답 확률 : [0.665 (A 의류), 0.665 (B 식품), 0.844 (C 가전)]  
상품별 CE : [−ln(0.665), −ln(0.665), −ln(0.844)] = [0.408, 0.408, 0.170]  
평균 CE : (0.408 + 0.408 + 0.170) / 3 약 0.328  

**Q3. 세 상품 모두 정답을 가장 높게 예측했는데 C의 손실이 더 작은 이유는 무엇인가요?**

- 답변: C가 정답 클래스에 더 높은 확률을 줬기 때문  
C가 실제 정답인 가전에 부여한 확률이 약 0.844로 높다. A와 B의 정답 확률은 약 0.665  
-> CE는 맞고 틀림이 아니라 실제 정답 확률을 반영하기 떄문에 C의 손실이 가장 작다.

**Q4. 점수 [2,1,0]을 곧바로 CE의 확률 자리에 넣어도 되나요?**

- 답변: 아니오  
[2,1,0]은 아직 확률이 아닌 점수이기 때문, 또한 0~1 사이가 아니고 합도 1이 아님 (2 + 1 + 0 = 3)  
-> 반드시 Softmax로 확률로 바꾼 뒤 정답 클래스의 확률을 CE에 넣어야 함

---

## 과제. 필수 실습과 같은 방법으로 복습하기

### 문제 3-1. 새로운 금액 예측의 MSE 복습하기

#### 문제 설명

교안 연습문제 1의 MSE 계산을 복습합니다. 금액 단위는 만 원입니다.

| 고객 | 실제값 | 모델 A 예측 | 모델 B 예측 |
|---|---:|---:|---:|
| 첫째 | 8 | 10 | 8 |
| 둘째 | 12 | 11 | 16 |
| 셋째 | 20 | 18 | 17 |

문제 1-1과 같은 순서로 계산하세요. 새 공식은 필요하지 않습니다.

#### 요구사항

1. 실제값과 두 예측 배열을 만드세요.
2. 각 모델의 오차와 제곱오차를 계산해 출력하세요.
3. SSE와 MSE를 구하고 MSE는 소수점 아래 세 자리로 출력하세요.
4. Q1~Q4에 답하세요.

#### 작성 도움말

`오차 = 실제값 − 예측값`이며, `MSE = np.mean(오차 ** 2)`입니다. 모델 A의 첫 오차는 8−10입니다. 평균의 분모는 고객 수 3입니다.

#### 해석 질문

**Q1.** 두 모델의 오차와 제곱오차는 각각 무엇인가요?

**Q2.** SSE와 MSE는 각각 얼마이며 어떤 모델을 선택하나요?

**Q3.** 모델 B는 첫 고객을 정확히 예측했는데도 전체 MSE가 더 큰 이유는 무엇인가요?

**Q4.** 모델 B의 MSE를 계산할 때 오차가 있는 두 고객만 세어 2로 나누어도 되나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [25]:
# 문제 3-1. 새로운 금액 예측의 MSE 복습하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1. 실제값과 두 예측 배열을 만드세요.
y = np.array([8, 12, 20])
pred_a = np.array([10, 11, 18])
pred_b = np.array([8, 16, 17])

# 2. 각 모델의 오차와 제곱오차를 계산해 출력하세요.
# 오차 = 실제값 − 예측값
error_a = y - pred_a
error_b = y - pred_b
print("모델 A 오차 :", error_a)
print("모델 B 오차 :", error_b)
print("모델 A 제곱 오차 :", error_a**2)
print("모델 B 제곱 오차 :", error_b**2)

# 3. SSE와 MSE를 구하고 MSE는 소수점 아래 세 자리로 출력하세요.
# SSE = 제곱오차를 전부 더한 값
# MSE = np.mean(오차 ** 2)
sse_a = np.sum(error_a**2)
sse_b= np.sum(error_b**2)
mse_a = np.mean(error_a**2)
mse_b = np.mean(error_b**2)
print("모델 A SSE :", sse_a)
print("모델 B SSE :", sse_b)
print("모델 A MSE :", np.round(mse_a, 3))
print("모델 B MSE :", np.round(mse_b, 3))

모델 A 오차 : [-2  1  2]
모델 B 오차 : [ 0 -4  3]
모델 A 제곱 오차 : [4 1 4]
모델 B 제곱 오차 : [ 0 16  9]
모델 A SSE : 9
모델 B SSE : 25
모델 A MSE : 3.0
모델 B MSE : 8.333


#### 답변 작성란

**Q1.** 두 모델의 오차와 제곱오차는 각각 무엇인가요?

- 모델 A  
오차 : [8 − 10, 12 − 11, 20 − 18]  
-> [−2, 1, 2]  
제곱오차 : [(−2)², 1², 2²]  
-> [4, 1, 4]  
- 모델 B  
오차 : [8 − 8, 12 − 16, 20 − 17]  
-> [0, −4, 3]  
제곱오차 : [0², (−4)², 3²]  
-> [0, 16, 9]  

**Q2.** SSE와 MSE는 각각 얼마이며 어떤 모델을 선택하나요?

- **모델 A**  
모델 A  
SSE : 4 + 1 + 4 = 9  
MSE : 9 / 3 = 3.0  
모델 B  
SSE : 0 + 16 + 9 = 25  
MSE : 25 / 3 = 8.333  
-> MSE가 더 작은 모델 A를 선택  

**Q3.** 모델 B는 첫 고객을 정확히 예측했는데도 전체 MSE가 더 큰 이유는 무엇인가요?

- 답변: 둘째, 셋째 고객의 오차가 크기 때문  
모델 B의 오차 −4, 3은 제곱하면 16, 9로 늘어남  
-> 오차를 제곱하므로 큰 오차일수록 벌점을 더 많이 부여  
-> 모델 A는 모든 고객에서 오차가 1~2로 모델 B보다 작아 제곱해도 4 이하임   

**Q4.** 모델 B의 MSE를 계산할 때 오차가 있는 두 고객만 세어 2로 나누어도 되나요?

- 답변: 아니요  
오차가 0인 첫째 고객도 모델이 예측한 고객이므로 평균에 포함시켜야 함  
-> MSE의 분모는 전체 고객 수인 3이어야 함

### 문제 3-2. 새로운 고객 데이터의 BCE 복습하기

#### 문제 설명

교안 연습문제 2의 고객 3명입니다. 실제값은 1=구매, 0=미구매입니다. 표의 확률은 구매 확률입니다.

| 고객 | 실제값 y | 모델 A 구매 확률 | 모델 B 구매 확률 |
|---|---:|---:|---:|
| 첫째 | 1 | 0.8 | 0.6 |
| 둘째 | 0 | 0.3 | 0.6 |
| 셋째 | 1 | 0.9 | 0.7 |

문제 1-2에서 사용한 BCE 함수를 그대로 사용합니다.

#### 요구사항

1. 실제값과 두 모델의 확률 배열을 만드세요.
2. 아래 함수를 이용해 각 모델의 고객별 BCE를 계산하세요.
3. `np.mean()`으로 두 평균 BCE를 구하세요. 고객별·평균 BCE는 소수점 아래 세 자리로 출력합니다.
4. Q1~Q4에 답하세요.

#### 제공 함수와 작성 도움말

```python
def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))
```

실제 정답이 0인 둘째 고객은 미구매 확률을 사용합니다. 모델 A에서는 1−0.3, 모델 B에서는 1−0.6입니다. 평균의 분모는 고객 수 3입니다.

#### 해석 질문

**Q1.** 두 모델이 실제 정답에 부여한 확률은 각각 무엇인가요?

**Q2.** 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?

**Q3.** 모델 B가 첫째와 둘째에게 같은 구매 확률 0.6을 주었는데도 BCE가 다른 이유는 무엇인가요?

**Q4.** 평균 BCE 기준으로 어느 모델을 선택하며, 그 이유는 무엇인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [29]:
# 문제 3-2. 새로운 고객 데이터의 BCE 복습하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1. 실제값과 두 모델의 확률 배열을 만드세요.
y = np.array([1, 0, 1])
pred_a = np.array([0.8, 0.3, 0.9])
pred_b = np.array([0.6, 0.6, 0.7])

# 2. 아래 함수를 이용해 각 모델의 고객별 BCE를 계산하세요.
def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))
bce_a = binary_cross_entropy(y,pred_a)
bce_b = binary_cross_entropy(y,pred_b)
print("모델 A 고객별 BCE :", np.round(bce_a, 3))
print("모델 B 고객별 BCE :", np.round(bce_b, 3))


# 3. `np.mean()`으로 두 평균 BCE를 구하세요. 고객별·평균 BCE는 소수점 아래 세 자리로 출력합니다.
mean_a = np.mean(bce_a)
mean_b = np.mean(bce_b)
print("모델 A 평균 BCE :", np.round(mean_a, 3))
print("모델 B 평균 BCE :", np.round(mean_b, 3))

모델 A 고객별 BCE : [0.223 0.357 0.105]
모델 B 고객별 BCE : [0.511 0.916 0.357]
모델 A 평균 BCE : 0.228
모델 B 평균 BCE : 0.595


#### 답변 작성란

**Q1.** 두 모델이 실제 정답에 부여한 확률은 각각 무엇인가요?

- 모델 A : [0.8 (구매), 1 − 0.3 (미구매), 0.9 (구매)]  
-> [0.8, 0.7, 0.9]  
- 모델 B : [0.6 (구매), 1 − 0.6 (미구매), 0.7 (구매)]  
-> [0.6, 0.4, 0.7]  

**Q2.** 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?

- 모델 A  
    - 고객별 BCE : [−log(0.8), −log(0.7), −log(0.9)]  
-> [0.223, 0.357, 0.105]  
    - 평균 BCE : (0.223 + 0.357 + 0.105) / 3  
-> 0.228  
- 모델 B  
    - 고객별 BCE : [−log(0.6), −log(0.4), −log(0.7)]  
    -> [0.511, 0.916, 0.357]  
    - 평균 BCE : (0.511 + 0.916 + 0.357) / 3  
    -> 0.595  

**Q3.** 모델 B가 첫째와 둘째에게 같은 구매 확률 0.6을 주었는데도 BCE가 다른 이유는 무엇인가요?

- 답변: 두 고객의 실제 정답이 다르기 때문  
BCE는 실제 정답에 부여한 확률로 계산하기 때문에 같은 구매 확률을 예측했더라도 정답에 따라 사용하는 확률이 달라질 수 있음  
    - 첫째 고객 : 구매(1)이므로 구매 확률 0.6 사용 -> BCE 0.511  
    - 둘째 고객 : 미구매(0)이므로 미구매 확률 1 − 0.6인 0.4 사용 -> BCE 0.916  
        -> 둘째 고객은 정답(미구매)에 0.4를 줘서 벌점이 더 큼  

**Q4.** 평균 BCE 기준으로 어느 모델을 선택하며, 그 이유는 무엇인가요?

- 답변: **모델 A**  
모델 A 평균 BCE : 0.228  
모델 B 평균 BCE : 0.595  
-> BCE는 작을수록 실제 정답에 높은 확률을 준 것이므로, 평균 BCE가 더 작은 모델 A를 선택  
-> Q1에서 모델 A는 세 고객 모두 실제 정답에 더 높은 확률(0.8, 0.7, 0.9)을 부여해 모델 B(0.6, 0.4, 0.7)보다 손실이 작음

---

## 실습 마무리

다음 질문으로 복습하세요. 6번은 교안의 연결 원리를 읽고 확인하는 질문입니다.

**1.** MSE를 구하는 계산 순서는 무엇인가요?  
-> 1-1. 실제값 - 예측값  
-> 1-2. 각 오차를 제곱  
-> 1-3. 제곱 오차를 모두 더함  
-> 1-4. 제곱 오차를 모두 더한 결과에 데이터 수로 나눔  
-> 제곱 오차의 합을 SSE라고 부르고, 평균은 MSE라고 부른다.

**2.** 실제값 10, 예측값 12일 때 오차 −2의 뜻은 무엇인가요?  
-> 실제보다 2만큼 높게 예측했다는 뜻  
-> 오차는 10 - 12로 -2고 제곱오차는 4

**3.** 실제 미구매인데 구매 확률이 0.7이면 BCE에 어떤 확률을 쓰나요?  
-> 미구매 확률 1-p인 1 - 0.7 = 0.3을 사용한다.  
-> BCE는 -ln(0.3) = 약 1.20

**4.** 상품 CE에서는 어떤 확률을 고르나요?  
-> 각 상품의 실제 정답 클래스에 해당하는 확률을 고른다.  
-> 모델이 가장 높게 예측한 클래스가 실제 정답과 다르더라도 기준은 실제 정답

**5.** 상품별 CE를 구한 다음 평균은 무엇으로 나누나요?  
-> 상품별 CE를 모두 더한 뒤 상품 수로 나눈다.  
-> 단, 클래스 수로 추가로 나누지 않는다.

**6.** MSE와 Gaussian NLL, BCE·CE와 NLL은 각각 어떻게 연결되나요?  
-> 독립적인 정규 분포 오차와 동일한 고정 분산을 가정하여 평균 gaussian NLL같은 같은 상수에 양의 상수 X MSE를 더한 형태  
-> 값 자체는 달라도 같은 최적 평균 파라미터를 선택  
-> 한 데이터의 BCE, CE는 각각 베르누이 범주 분포의 NLL  
-> 조건부 독립 가정에서 데이터별 손실을 더하면 전체 NLL이고, 데이터 수로 나누면 BCE, CE이다. 

In [ ]:
# 손실 함수와 Loss
# 모델의 예측을 실제 정답과 비교하여 숫자로 평가하는 규칙
# 금액 예측은 실제값과 예측값의 차이로 MSE를 구할 수 있다.
# 구매 여부와 상품 종류는 실제 정답에 주어진 확률로 BCE 또는 CE를 구할 수 있다
# 문제 종류에 맞게 계산 규칙을 선택한다.
# ex) 배달 시간 예측 모델 두 개를 비교하겠다고 가정해보면
# 실제 시간은 [10,20,30]분이고, 예측 시간은 [12,18,27]분이라면
# 각 주문에서 실제 시간과 예측 시간의 차이를 제곱하면 [4,4,9]다. <- 평균을 계산해보면 17/3 = 5.66
# 같은 주문에서 B의 평균제곱오차가 8.667이라면 5.66인 A모델이 더 좋았다고 평가할 수 있다.
# 손실함수는 모델의 예측을 비교하고 학습 중 어떤 것을 줄여야하는지 정하는 역할을 한다.

# 제곱오차합과 SSE (Sum of Squared Erros)
# 제곱 오차를 모두 더한 값
# 제곱오차 [4,4,9]ㅣ의 SSE는 17이다.
# 배달 3건에 대해서 A모델의 제곱오차가 [4,4,9], B모델의 제곱오차가 [1,16,9]라고 가정
# A의 합계는 17, B는 26 <- 이 합게가 의미하는 것이 SSE
# 같은 세 건에 대해서는 A의 전체 제곱오차가 더 작다.
# B가 비록 첫 주문을 A모델보다 더 잘 맞췄더라도 전체 합계에서는 A가 더 좋은 모델일 수 있다.
# SSE를 전체 주문 건수로 나누면 MSE가 된다.
# 그러면 이걸 왜 하느냐?
# -> SSE를 통해 MSE를 구할 수 있고, 합계가 커진 이유 자체가 오차인지, 데이터 수에 영향을 받았는지 구분할 수 있음

# 평균제곱오차 MSE (Mean Squared Error)
# 제곱오차의 평균
# 각 고객의 오차를 제곱하고 모두 더한 뒤 전체 고객 수로 나눈다.
# 배달 세 건의 실제 시간이 [10,20,30]분이고 예측 시간은 [12,18,27]분이라면
# 실제 예측은 [-2,2,3], 각 오차의 제곱은 [4,4,9] 합계 17을 3(전체 고객 수)으로 나누면 5.66
# 다른 B모델의 MSE가 8.66이라면 A모델을 선택하는 것이 맞다.
# 그렇다면 왜 중요하게 배우는 건가?
# 여러 수치의 예측 제곱오차를 하나의 평균으로 요약해 비교하기 위해서다.
# 오차를 전체 수로 평균을 내기 때문에 각 한 건에 대한 오차범위를 대략적으로 파악할 수 있다.

# MSE의 단위
# 오차를 제곱하면 단위도 제곱하면 된다.
# 단위가 만원 단위였다면 제곱오차에 대한 해석도 (만원)² 단위로 읽으면 된다.

# 이진 교차엔트로피와 BCE
# Binary Cross Entropy, 구매 / 미구매 두 가지의 결과를 구분하는 분류 손실함수
# 한 고객의 BCE는 실제 정답 확률에 음의 로그를 적용한 값
# ex) 두 예약자에 실제 방문 결과가 [1,0]이고, A의 방문 확률이 [0.8, 0.3]이라고 가정하면
# 첫번째 손님의 손실은 -in(0.8) = 약 0.223, 둘째는 미방문이니까 -in(0.7) = 약 0.356정도
# 평균 BCE값은 약 0.289
# B가 두 사람 모두에게 방문 확률 0.5를 주었다면 두 손실은 각각 약 0.6931로 평균도 0.6931로 나타난다.
# 같은 두 예약 확률에서는 BCE가 작은 A모델을 선택한다.
# 왜 쓰냐?
# 두 예측 결과를 학습하고 평가할 기준으로 쓴다.
# 각 정답에 어느 정도의 확률을 부여했는지 알 수 있다.
# 두 값 중 더 작은 숫자가 더 적은 손실을 냈다고 알면 됨

# 범주형 교차 엔트로피와 CE (Categorical Cross Entropy)
# 여러 종류중 실제 정답이 하나인 분류에 쓰이는 손실
# 상품 B의 정답이 수박 [배,수박,사과] 확률이 [0.2, 0.6, -.2]라면 CE = in(0.6) = 약 0.511
# ex) 과일 종류를 [배,수박,사과] 순서로 분류한다고 했을 때
# 첫 문의의 실제 정답은 수박이고 예측 확률이 [0.2,0.6,0.2]일 때 정답인 수박의 0.6을 선택하여
# -ln(0.6) = 약 0.511을 계산한다.
# 두번째 정답 확률이 0.8이라면 CE = 0.2231, 두 과일 선택의 평균을 내보면 0.367정도
# 왜 쓰느냐, 과일 선택처럼 대상마다 열의 확률을 고르고 대상별 손실을 평균내는 계산을 활용할 수 있다.

# 예측 클래스와 손실의 차이
# 예측 클래스 : 모델이 최종적으로 선택한 종류
# 손실 : 실제 정답에 준 확률을 평가한 값
# 정답을 맞추면 손실이 0이냐? -> 아님
# ex) 실제 구매 고객에게 확률 0.9와 0.6을 주면 임계값 0.5에서 둘 다 구매로 예측
# 하지만 BCE는 약 0.105와 0.511로 다를 수 있음
# -> 정답을 맞췄다는 사실만으로 예측의 품질을 알 수가 없기 때문에 손실 점수를 확인해야한다.
# 두 가지 사실, 정답을 맞췄는가? 확률 예측 자체가 얼마나 좋은가?에 대한 답을 할 수 있다.

# 정규분포(가우시안)
# 평균을 중심으로 좌우가 대칭인 종 모양의 연속형 분포
# ex) 배달 모델에서 주문 도착 시간을 20분으로 예상했다고 가정
# 실제 배달 시간이 19분, 21분처럼 약간의 오차가 발생할 수는 있다.
# 비슷한 조건에서 주문에서 작은 차이는 자주, 큰 차이는 드물에 나타난다고 설명하기 위해
# 정규분포를 가정할 수 있다.

In [ ]:
# 작업시간으로 생산량 예측
# 과거에 2시간 작업해서 10개를 만든 기록이 있다고 가정
# 입력 x는 2시간, y는 10
# 처음 모델에는 시간당 3개를 만드는 모델을 설계한다고 가정
# y=wx + b
# y = 3x2 + 1 = 7
# 입력 2 -> 주어진 작업 기록으로 고정
# 정답 10 -> 비교 기준으로 고정
# 가중치 w, 3 ->학습하여 조절할 값
# 예측값 ŷ -> w를 바꾸면서 계산

# 손실은 모델이 예측값과 실제값 사이에서 틀린 정도를 나타냄
# 보통 차이를 계산하고 제곱
# L(w) = (y - y_pred)² = (10 - 2w)²
# L(3) = (10 - 6)² = 16
# X = 2, Y = 10, W = 3 이라면 에측값은 6, 정답과 예측의 차이는 4
# 그냥 4라고 하면 되는데 왜 제곱?
# -> 정답 -4만큼, +4만큼이 구분되어야하기 때문에 제곱을 활용 (방향을 알아야하기 때문)
# 손실은 어느정도 틀렸는지를 알려주고, 어느 방향으로 수정해야하는지는 확인이 따로 필요
# 어느 방향으로 수정해야하는지는 미분을 통해 알 수 있다.

# 변화율 : 변한 양을 비교
# f(x) = 2x + 1
# 어느 구간을 선택해도 기울기가 항상 일정한 직선이 된다.


# 미분은 지금 위치의 변화율을 나타냄
# 같은 1만큼 움직여도 결과가 달라짐
# 한 점을 찍었을 때 순간 변화율을 구하는 것이 미분

# 필요한 미분 공식
# 변화가 없으면 기울기 0
# f(x) = 5면 x에 1, 100, 1,000을 넣어도 결과는 5
# x를 바꿔도 출력이 바뀌지 않으면 도함수는 0
# f(x) = 3x + 5에서는 x가 1 증가할 때마다 결과가 3씩 증가한다. <- 도함수는 3
# +5는 그래프의 전체 높이를 높일 뿐 기울기를 바꾸지 않는다

# 편미분 : 하나씩 바꾸어보는 것을 의미

In [3]:
# 입력과 정답은 고정

x = 2
y = 10
w = 3.0
b = 1.0
learning_rate = 0.01

# 1. 현재 예측과 손실을 계산
y_pred = w * x + b
old_loss = ( y - y_pred) ** 2

# 2. 같은 현재 w,b에서 기울기 구하기
grad_w = 8 * w + 4 * b - 40
grad_b = 4 * w + 2 * b - 20

# 3. 기울기의 반대방향으로 값을 바꾸기
new_w = w - learning_rate * grad_w
new_b = b - learning_rate * grad_b


# 4. 바꾼 값으로 예측과 손실을 다시 계산
new_pred = new_w * x + new_b
new_loss = ( y - new_pred)
print("기울기", grad_w, grad_b)
print("새 w와 b", new_w, new_b)
print("새 예측", round(new_pred, 2))
print("손실", old_loss, "->", round(new_loss, 2))

기울기 -12.0 -6.0
새 w와 b 3.12 1.06
새 예측 7.3
손실 9.0 -> 2.7


In [4]:
# 회귀 모델의 순전파를 따라가기
# 2시간 작업의 생산량을 예측하기
# 입력 x = 2는 작업시간, 실제 정답 y = 10을 생산량이라고 가정
# 현재 모델은 w = 3, b = 1을 사용
# u = wx = 3 x 2 = 6
# y_pred = u + b = 6 + 1 = 7
# L = (y - ypred)² = 9
# 순전파는 입력에서 시작하여 예측을 구하는 과정을 의미

| 구분 | 현재 저장할 값 | 역전파에서 필요한 이유 |
|---|---|---|
| 입력 | x = 2 | w의 국소 미분값으로 사용 |
| 곱셈 결과 | u = 6 | 계산이 어떻게 이어졌는지 확인 |
| 예측 | y_pred = 7 | 손실의 미분값을 구할 때 활용 |
| 정답 | y = 10 | 예측과 비교할 기준 |

각 중간값에 대한 손실의 기울기를 계산하여 역전파에서 전달  
- 손실에서 시작한다면 어떤 것을 미분해야 하는가?
- 예측값이 조금 달라지면 손실은 어떻게 달라지는가?

가중치 편향 고려하지 않고 y_pred의 값이 조금 바뀐다고 가정

| 예측값 | 손실 계산 | 손실 |
|---|---|---|
| 7.00 | (10 − 7.00)² | 9 |
| 7.01 | (10 − 7.01)² | 8.940 |

예측을 0.01 늘렸더니 손실은 0.059 감소, 현재 기울기 -6으로 예상한 변화 -6 x 0.01에 가까운 수치

가중치와 편향까지 기울기 전달하기  
앞서 기울기 -6, ∂L/∂ŷ=−6의 계산식을 얻었다.  
ŷ = wx + b에서 w와 b가 각각 무엇을 바꿀 수 있는지?( x=2 고정 )  
가중치 w를 0.01 늘리면 예측이 2 x 0.01 = 0.02 늘어남  
따라서 w에 대한 예측의 변화율은 2, 편향 b도 0.01 늘리면 예측도 0.01 늘어나기 때문에 변화율은 1  
∂ŷ/∂w = x = 2, / ∂ŷ/∂b = 1  
-6 x 2 = -12와 -6 x 1 = -6값을 얻을 수 있는데  

| 현재 계산 대상 | 값 | 의미 |
|---|---|---|
| 예측 ŷ | 7 | 모델이 내놓은 답 |
| 손실 L | 9 | 예측이 틀린 정도 |
| ∂L/∂ŷ | −6 | 예측을 조금 바꿀 때 손실의 변화율 |
| ∂L/∂w | −12 | w를 조금 바꿀 때 손실의 변화율 |
| ∂L/∂b | −6 | b를 조금 바꿀 때 손실의 변화율 |

L(w, b) = (10 − 2w − b)²를 전개하여 미분하면

- ∂L/∂w = 8w + 4b − 40
- ∂L/∂b = 4w + 2b − 20

역전파 다음에 가중치를 바꾼다.  
기울기를 계산하는 일 / 적용하는 일 분리  
역전파로 gradient [-12, -6]을 얻었지만, 여전히 w와 b는 3,1로 고정인 상태  
이 3, 1을 업데이트 하는 규칙을 실행해야 한다.  
새로운 파라미터 - 현재 파라미터 - 학습률 x 기울기  
학습률 - 한 번에 학습률만큼 크기를 조절하겠다는 의미  
업데이트를 진행한 뒤에는 다시 순전파를 시작함  
새w = 3 x 0.01 x (-12) = 3.12  
새b = 1-0.01 x (-6) = 1.06  
새 y_pred = 3.12 x 2 + 1.06 = 7.30  
새 L = (10 - 7.30)² = 7.29  

| 단계 | 계산하는 내용 | 이번 결과 |
|---|---|---|
| 순전파 | 현재 파라미터로 예측 | 7 |
| 손실 계산 | 예측과 정답의 차이 평가 | 9 |
| 역전파 | w와 b의 기울기 | −12와 −6 |
| 업데이트 | 기울기로 파라미터 변경 | 3.12와 1.06 |
| 다음 순전파 | 바뀐 모델로 다시 계산 | 예측 7.30과 손실 7.29 |

-> 왜 학습률이 0.001처럼 낮은 숫자일까?  
현재 기울기는 지금 위치의 근처 정보, 한 번에 너무 크게 바꾸면 손실이 갑자기 커질 수 있음


In [8]:
# 코드로 회귀 역전파 확인하기
# x = 3, y = 15, 
x, y = 3.0, 15.0
w, b = 3.0, 2.0
learning_rate = 0.01

# 1. 순전파의 손실계산
y_pred = w * x + b # <- 레이어 1
loss = (y - y_pred) ** 2

# 2. 손실에서 예측값으로(역전파)
dL_dy_pred = 2 * (y_pred - y)

# 3. 예측값에서 각 파라미터로(역전파)
dL_dw = dL_dy_pred * x
dL_db = dL_dy_pred * 1.0
print("에측과 손실", y_pred, loss)
print("기울기", dL_dy_pred, dL_dw, dL_db)
print("업데이트 전", w, b)

new_w = w - learning_rate * dL_dw
new_b = b - learning_rate * dL_db
new_pred = new_w * x + new_b
new_loss = ( y - new_pred) ** 2
print("업데이트 후", new_w, new_b)
print("새 손실", round(new_loss, 2))

에측과 손실 11.0 16.0
기울기 -8.0 -24.0 -8.0
업데이트 전 3.0 2.0
업데이트 후 3.24 2.08
새 손실 10.24


In [14]:
import numpy as np
# 코드로 은닉층과 역전파 확인하기
x, y = 2.0, 1.0
w1, b1 = 1.0, 0.0
w2, b2 = 0.5, 0.0

#### 1.순전파
a = w1 * x + b1 
# 레이어1 아핀 변환 결과 (ReLU 전)

h = max(0.0, a) # <-여기까지 레이어 1 (뉴런 1개의 계산), 은닉층의 뉴런 1개가 입력을 받아 출력값 h을 만듦
# 레이어1 최종 출력 (ReLU 후) 은닉층 출력

# 뉴런이 3개였는데 왜 한 개만 계산? -> 같은 입력 x를 받지만 뉴런은 자기 가중치와 자기 편향으로 따로 계산 
# 은닉층 하나에 뉴런 3개 h1 = max(0.0, w1 * x + b1) h2 = max(0.0 w2 * x + b2)
# -> 은닉층의 출력이 [h1, h2, h3] 앞에 뉴런을 차례대로 계산하는 게 하나라 같은 층에서 나란히 계산하기 때문


z = w2* h + b2
# 레이어2 아핀 변환 결과 (sigmoid 들어가기 전)

p = 1.0 / (1.0 + np.exp(-z)) # <- 여기까지 레이어2
# 레이어2 최종 출력 (정답이 1일 확률)

loss = np.logaddexp(0.0, z) - y * z # <- 손실은 레이어 아님
# 손실 (예측이 틀린 정도, BCE)


#### 2. 출력층의 역전파 (손실은 레이어 x, 출력층의 기울기 계산)
dL_dz = p - y
# 예측 확률(p) − 정답(y)
# -> z를 바꾸면 손실이 얼마나 변하나? (출발점)
# sigmoid + BCE를 체인룰로 곱하면 p − y로 정리됨

dL_dw2 = dL_dz * h
# dL_dz × h (레이어1의 ReLU 후 값)
# 출력층 가중치 w2의 기울기

dL_db2 = dL_dz
# dL_dz × 1 (편향은 더해지기만 하므로 기울기 1)
# 출력층 편향 b2의 기울기

# ReLU를 거쳐 은닉층으로 전달 (은닉층 기울기 계산)
dL_dh = dL_dz * w2
# dL_dz × w2
# 은닉층 출력 h의 기울기 (1층으로 전달할 값)

relu_grad = 1.0 if a > 0 else 0.0
# a = 2 > 0 이니까 1
# ReLU의 기울기 (통과시키면 1, 막으면 0)

dL_da = dL_dh * relu_grad
# dL_dh × relu_grad
# ReLU 들어가기 전 a의 기울기

dL_dw1 = dL_da * x
# dL_da × x (입력값)
# 레이어1 가중치 w1의 기울기

dL_db1 = dL_da
# dL_da × 1 (편향은 더해지기만 하므로 기울기 1)
# 레이어1 편향 b1의 기울기

print("중간값", a, h, z)
print("확률과 손실", round(p,4), round(loss, 4) )
print("첫 층", round(dL_dw1, 4), round(dL_db1, 4))
print("출력층", round(dL_dw2, 4), round(dL_db2, 4))

중간값 2.0 2.0 1.0
확률과 손실 0.7311 0.3133
첫 층 -0.2689 -0.1345
출력층 -0.5379 -0.2689
